# 01 — Exploratory Data Analysis
**Dataset:** Road Traffic Accidents — Addis Ababa Sub-City Police Department  
**Rows:** 12,316 | **Features:** 31 | **Target:** Accident_severity (3 classes)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

DATA_PATH = Path('../data/RTA_Dataset.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Target distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Accident_severity'].value_counts()
colors = ['#16a34a', '#d97706', '#dc2626']

axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Severity')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 50, f'{count:,}\n({count/len(df)*100:.1f}%)', ha='center', fontsize=9)

axes[1].pie(counts.values, labels=counts.index, colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Distribution (Proportion)')

plt.tight_layout()
plt.show()
print('Class imbalance confirmed — SMOTE required')

In [ ]:
# ── Missing values ───────────────────────────────────────────────────────────
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

if len(null_pct) > 0:
    plt.figure(figsize=(10, 4))
    null_pct.plot(kind='bar', color='#6366f1')
    plt.title('Missing Value % per Column')
    plt.ylabel('% Missing')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found!')

print(null_pct)

In [ ]:
# ── Cause of accident vs severity ────────────────────────────────────────────
cause_severity = pd.crosstab(
    df['Cause_of_accident'], df['Accident_severity'], normalize='index'
) * 100

cause_severity.plot(kind='bar', stacked=True, figsize=(14, 5),
                    color=['#16a34a', '#d97706', '#dc2626'])
plt.title('Cause of Accident → Severity Distribution (%)')
plt.xlabel('Cause of Accident')
plt.ylabel('% of accidents')
plt.legend(title='Severity', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Hour of day distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hour_counts = df['Hour_of_day'].value_counts().sort_index()
axes[0].plot(hour_counts.index, hour_counts.values, 'o-', color='#6366f1')
axes[0].fill_between(hour_counts.index, hour_counts.values, alpha=0.2, color='#6366f1')
axes[0].set_title('Accidents by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_xticks(range(0, 24))

hour_severity = pd.crosstab(df['Hour_of_day'], df['Accident_severity'])
hour_severity.plot(ax=axes[1], color=['#16a34a', '#d97706', '#dc2626'])
axes[1].set_title('Severity by Hour of Day')
axes[1].set_xlabel('Hour')

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap (after encoding) ─────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy().dropna()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

corr = df_enc.corr()['Accident_severity'].drop('Accident_severity').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 6))
corr.head(15).plot(kind='barh', color=['#dc2626' if v > 0 else '#2563eb' for v in corr.head(15)])
plt.title('Top 15 Features Correlated with Accident Severity')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary stats ────────────────────────────────────────────────────────────
print('=== Dataset Summary ===')
print(f'Total rows:       {len(df):,}')
print(f'Total columns:    {len(df.columns)}')
print(f'Numeric columns:  {df.select_dtypes(include=np.number).shape[1]}')
print(f'Object columns:   {df.select_dtypes(include=object).shape[1]}')
print(f'Null values:      {df.isnull().sum().sum():,}')
print()
print('=== Target Distribution ===')
for label, count in df['Accident_severity'].value_counts().items():
    print(f'  {label:<20}: {count:>5,}  ({count/len(df)*100:.1f}%)')